# Student Performance Data Analysis
**Task 3 – Real-World Dataset Analysis Project**  
**Prepared by: Nandigam Praveen Kumar**  
**Internship: YuvaIntern – Data Analyst**

## 1. Project objective
Analyze the UCI Student Performance dataset to explore how study time, absences, previous failures, family/education factors and period grades relate to final academic performance (G3). This is an exploratory analysis, not a causal study or a validated student-risk scoring system.

## 2. Dataset source
UCI Machine Learning Repository, **Student Performance (Dataset ID: 320)**. The dataset was collected from two Portuguese secondary schools and includes student grades, demographic, social and school-related attributes. This notebook uses the **Portuguese-language course file** (649 records) for one consistent analysis.

Official source: https://archive.ics.uci.edu/dataset/320/student-performance

The UCI description notes that G1 and G2 are earlier-period grades and are strongly associated with G3. They are included in descriptive analysis, but must be excluded from any early-warning model intended to predict final grades before those marks are available.


## 3. Install and import required libraries
Run this cell in Google Colab or Jupyter. The `ucimlrepo` package retrieves the dataset from UCI.

In [ ]:
%pip -q install ucimlrepo

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

## 4. Load the real-world dataset

In [ ]:
student_performance = fetch_ucirepo(id=320)

X = student_performance.data.features.copy()
y = student_performance.data.targets.copy()

# UCI provides separate Mathematics and Portuguese course files in the original dataset.
# The repository interface exposes the Portuguese course dataset for this selected import.
df = pd.concat([X, y], axis=1)

print("Dataset shape:", df.shape)
display(df.head())


## 5. Understand the dataset structure

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("data_type"))
print("\nDataset information:")
df.info()
print("\nDescriptive statistics:")
display(df.describe(include="all").T)


## 6. Data quality assessment

In [ ]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})
display(quality)

duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows:", duplicate_count)


### Data cleaning and preprocessing

In [ ]:
# Preserve the raw dataframe and create a working copy.
clean_df = df.copy()

# Standardize column names for easier coding.
clean_df.columns = [str(c).strip() for c in clean_df.columns]

# Remove exact duplicate rows only if any are present.
before = len(clean_df)
clean_df = clean_df.drop_duplicates().copy()
print("Rows before:", before, "| Rows after duplicate check:", len(clean_df))

# Convert numeric-looking columns safely where appropriate.
for col in clean_df.columns:
    if clean_df[col].dtype == "object":
        # Do not force categorical text columns into numbers.
        clean_df[col] = clean_df[col].astype(str).str.strip()

print("Remaining missing values:", int(clean_df.isna().sum().sum()))
print("Cleaned shape:", clean_df.shape)


## 7. Exploratory Data Analysis (EDA)

In [ ]:
# Identify the final-grade column.
grade_col = "G3"
if grade_col not in clean_df.columns:
    raise KeyError("Expected final grade column 'G3' was not found. Inspect the UCI column names.")

clean_df[grade_col] = pd.to_numeric(clean_df[grade_col], errors="coerce")
display(clean_df[grade_col].describe())

print("Mean final grade:", round(clean_df[grade_col].mean(), 2))
print("Median final grade:", round(clean_df[grade_col].median(), 2))
print("Standard deviation:", round(clean_df[grade_col].std(), 2))


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(clean_df[grade_col].dropna(), bins=15, kde=True)
plt.title("Distribution of Final Grades (G3)")
plt.xlabel("Final grade (0–20)")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()


In [ ]:
# Study-time group vs final grade
if "studytime" in clean_df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=clean_df, x="studytime", y=grade_col)
    plt.title("Final Grade by Weekly Study-Time Category")
    plt.xlabel("Study time category (UCI coded scale)")
    plt.ylabel("Final grade (G3)")
    plt.tight_layout()
    plt.show()
    display(clean_df.groupby("studytime")[grade_col].agg(["count", "mean", "median"]).round(2))


In [ ]:
# Absences vs final grade
if "absences" in clean_df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=clean_df, x="absences", y=grade_col, alpha=0.65)
    plt.title("Absences vs Final Grade")
    plt.xlabel("Number of absences")
    plt.ylabel("Final grade (G3)")
    plt.tight_layout()
    plt.show()
    print("Correlation (absences, G3):", round(clean_df[["absences", grade_col]].corr().iloc[0, 1], 3))


In [ ]:
# Prior failures vs final grade
if "failures" in clean_df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=clean_df, x="failures", y=grade_col)
    plt.title("Final Grade by Number of Previous Failures")
    plt.xlabel("Previous class failures (UCI-coded)")
    plt.ylabel("Final grade (G3)")
    plt.tight_layout()
    plt.show()
    display(clean_df.groupby("failures")[grade_col].agg(["count", "mean", "median"]).round(2))


## 8. Relationships among numerical variables

In [ ]:
numeric_df = clean_df.select_dtypes(include=np.number)
corr = numeric_df.corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of Numerical Variables")
plt.tight_layout()
plt.show()

if grade_col in corr.columns:
    display(corr[grade_col].sort_values(ascending=False).to_frame("correlation_with_G3"))


## 9. Categorical comparisons

In [ ]:
for col in ["sex", "address", "schoolsup", "famsup", "internet", "higher"]:
    if col in clean_df.columns:
        print(f"Average final grade by {col}:")
        display(clean_df.groupby(col, dropna=False)[grade_col].agg(["count", "mean", "median"]).round(2))


## 10. Automatically generated observations

In [ ]:
observations = []

if "studytime" in clean_df.columns:
    study_means = clean_df.groupby("studytime")[grade_col].mean().dropna()
    if len(study_means) >= 2:
        observations.append(
            f"Mean G3 varies across study-time categories; lowest category mean = {study_means.min():.2f}, "
            f"highest category mean = {study_means.max():.2f}. This is an association, not proof of causation."
        )

if "absences" in clean_df.columns:
    r = clean_df[["absences", grade_col]].corr().iloc[0, 1]
    observations.append(f"Pearson correlation between absences and G3 = {r:.3f}. Interpret its size and direction cautiously.")

if "failures" in clean_df.columns:
    fail_means = clean_df.groupby("failures")[grade_col].mean().dropna()
    if len(fail_means) >= 2:
        observations.append(
            f"Mean G3 ranges from {fail_means.min():.2f} to {fail_means.max():.2f} across previous-failure categories."
        )

for i, item in enumerate(observations, 1):
    print(f"{i}. {item}")

print("\nImportant limitation: these are observational data from two Portuguese schools, not a random sample of all students.")


## 11. Conclusions and practical recommendations
Use the numerical outputs and charts generated above to write the final findings. Do not claim that one factor causes higher or lower grades.

Potential recommendations to evaluate against the results:
1. Identify students with repeated previous failures for timely academic support.
2. Review attendance patterns and provide support where frequent absences coincide with weaker performance.
3. Encourage structured study habits while recognizing that study-time categories are self-reported/grouped measures.
4. Use earlier-period grades (G1/G2) only when they would genuinely be available at the prediction date.
5. Avoid using demographic or family attributes to label students; use analysis to improve support, not to discriminate.

## 12. Limitations
- The dataset represents two Portuguese schools and may not generalize to other countries or institutions.
- The data are observational; correlation does not establish causation.
- Some variables are ordinal categories encoded as numbers; treating them as continuous measurements can be misleading.
- G1 and G2 are closely related to G3 and can create information leakage in early prediction.
- Dataset records do not explain every contextual factor affecting student performance.

## 13. Export cleaned data and summary


In [ ]:
clean_df.to_csv("student_performance_cleaned.csv", index=False)
summary = clean_df.describe(include="all").T
summary.to_csv("student_performance_summary.csv")
print("Saved: student_performance_cleaned.csv")
print("Saved: student_performance_summary.csv")


## 14. References
- UCI Machine Learning Repository. Student Performance, Dataset ID 320. https://archive.ics.uci.edu/dataset/320/student-performance
- Cortez, P. & Silva, A. (2008). Using data mining to predict secondary school student performance. Proceedings of the 5th Annual Future Business Technology Conference.

**Prepared by:** Nandigam Praveen Kumar  
**Project:** Task 3 – Real-World Dataset Analysis Project  
